# EMIF Class 12 - Dynamic correlations, ERC portfolios and empirical betas

This notebook reproduces the empirical material used in the slides. It uses the multi-asset price file `Data_multiasset.xlsx`.

The goal is pedagogical: we build simple covariance estimates, ERC portfolios, a DCC-style correlation filter, and DCC-implied empirical betas. The code is intentionally transparent rather than industrial.


## 1. Load packages and data

Missing prices are carried forward. Because log returns require positive prices, non-positive observations are treated as missing and then carried forward. This matters for the oil futures series in 2020.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib.ticker import PercentFormatter

DATA_PATH = "Data_multiasset.xlsx"
ANNUALIZATION = 252

prices = pd.read_excel(DATA_PATH, sheet_name="Feuil1")
prices = prices.rename(columns={prices.columns[0]: "Date"}).set_index("Date")
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index().ffill()

# Log returns require strictly positive prices.
# Non-positive observations are treated as missing values and carried forward.
prices = prices.where(prices > 0).ffill().dropna(how="any")
returns = np.log(prices).diff().dropna()
assets = list(returns.columns)

prices.tail(), returns.tail()


## 2. Volatility and correlation move together

A covariance matrix is not just a list of individual variances. It also embeds co-movements. In stressed markets, correlations often rise while volatilities rise too, which reduces the ex-ante benefit of diversification.


In [ ]:
ROLLING_WINDOW = 252

rolling_vol = returns.rolling(ROLLING_WINDOW).std() * np.sqrt(ANNUALIZATION)
average_vol = rolling_vol.mean(axis=1)

average_corr = []
index = []
for i in range(ROLLING_WINDOW - 1, len(returns)):
    corr = returns.iloc[i - ROLLING_WINDOW + 1:i + 1].corr().values
    n = corr.shape[0]
    average_corr.append(corr[np.triu_indices(n, 1)].mean())
    index.append(returns.index[i])

average_corr = pd.Series(average_corr, index=index, name="Average pairwise correlation")

fig, ax1 = plt.subplots(figsize=(12, 6))
average_vol.plot(ax=ax1, label="Average annualized volatility")
ax1.set_ylabel("Average volatility")
ax1.yaxis.set_major_formatter(PercentFormatter(1.0))

ax2 = ax1.twinx()
average_corr.plot(ax=ax2, linestyle="--", label="Average pairwise correlation")
ax2.set_ylabel("Average correlation")

ax1.set_title("Volatility and correlation tend to rise together")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Equal Risk Contribution

For portfolio weights $w$ and covariance matrix $\Sigma$, portfolio volatility is

$$
\sigma_p = \sqrt{w'\Sigma w}.
$$

The marginal contribution of asset $i$ to portfolio volatility is

$$
rac{\partial \sigma_p}{\partial w_i} = rac{(\Sigma w)_i}{\sigma_p}.
$$

Its total contribution is

$$
RC_i = w_i rac{(\Sigma w)_i}{\sigma_p}.
$$

ERC looks for weights such that each asset contributes the same share of total portfolio risk.


In [ ]:
def portfolio_volatility(weights, cov):
    return float(np.sqrt(weights @ cov @ weights))

def risk_contributions(weights, cov):
    sigma = portfolio_volatility(weights, cov)
    marginal = cov @ weights / sigma
    total = weights * marginal
    percent = total / sigma
    return total, percent

def erc_weights(cov, start=None):
    n = cov.shape[0]
    cov = np.asarray(cov, dtype=float)
    cov = (cov + cov.T) / 2
    eig = np.linalg.eigvalsh(cov)
    if eig.min() <= 1e-12:
        cov = cov + np.eye(n) * (1e-8 - eig.min())
    if start is None:
        start = np.repeat(1 / n, n)

    bounds = [(0, 1)] * n
    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1},)

    def objective(w):
        _, percent = risk_contributions(w, cov)
        return np.sum((percent - 1 / n) ** 2)

    result = minimize(objective, start, method="SLSQP", bounds=bounds, constraints=constraints,
                      options={"maxiter": 1000, "ftol": 1e-12})
    if not result.success:
        result = minimize(objective, np.repeat(1 / n, n), method="SLSQP", bounds=bounds,
                          constraints=constraints, options={"maxiter": 2000, "ftol": 1e-12})

    w = np.maximum(result.x, 0)
    return w / w.sum()


## 4. Rolling ERC allocation

The covariance matrix is estimated on a 2-year rolling window. The ERC portfolio is rebalanced monthly.


In [ ]:
ROLLING_COV_WINDOW = 252 * 2
month_ends = returns.resample("M").last().index

weights = []
previous = None
for date in month_ends:
    if date < returns.index[ROLLING_COV_WINDOW]:
        continue
    loc = returns.index.searchsorted(date, side="right") - 1
    if loc < ROLLING_COV_WINDOW:
        continue
    cov = returns.iloc[loc - ROLLING_COV_WINDOW + 1:loc + 1].cov().values * ANNUALIZATION
    w = erc_weights(cov, previous)
    previous = w
    weights.append(pd.Series(w, index=assets, name=returns.index[loc]))

weights = pd.DataFrame(weights)
weights.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(weights.index, [weights[a] for a in assets], labels=assets)
ax.set_title("Equal Risk Contribution weights from rolling covariance matrix")
ax.set_ylabel("Portfolio weight")
ax.set_ylim(0, 1)
ax.legend(loc="upper left", ncol=3, fontsize=8)
plt.tight_layout()
plt.show()


## 5. Risk contributions

The next chart compares the risk contributions of an equal-weighted portfolio and an ERC portfolio using the latest rolling covariance matrix.


In [ ]:
latest_cov = returns.iloc[-ROLLING_COV_WINDOW:].cov().values * ANNUALIZATION
w_erc = weights.iloc[-1].values
w_equal = np.repeat(1 / len(assets), len(assets))

_, rc_erc = risk_contributions(w_erc, latest_cov)
_, rc_equal = risk_contributions(w_equal, latest_cov)

risk_contribution_table = pd.DataFrame({"Equal Weight": rc_equal, "ERC": rc_erc}, index=assets)

fig, ax = plt.subplots(figsize=(12, 6))
risk_contribution_table.plot(kind="bar", ax=ax)
ax.set_title("Risk contribution: Equal Weight vs ERC")
ax.set_ylabel("Share of portfolio risk")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Simple backtest

The comparison is deliberately simple: monthly ERC rebalancing versus equal weight over the same sample.


In [ ]:
daily_weights = weights.reindex(returns.index, method="ffill").dropna()
aligned_returns = returns.loc[daily_weights.index]

erc_returns = (daily_weights * aligned_returns).sum(axis=1)
equal_returns = aligned_returns.mean(axis=1)

cumulative = pd.DataFrame({
    "Equal Weight": (1 + equal_returns).cumprod(),
    "ERC": (1 + erc_returns).cumprod()
})

fig, ax = plt.subplots(figsize=(12, 6))
(cumulative / cumulative.iloc[0]).plot(ax=ax)
ax.set_title("Multi-asset performance: Equal Weight vs ERC")
ax.set_ylabel("Growth of 1")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def performance_statistics(r):
    ann_return = (1 + r).prod() ** (ANNUALIZATION / len(r)) - 1
    ann_vol = r.std() * np.sqrt(ANNUALIZATION)
    sharpe = ann_return / ann_vol
    wealth = (1 + r).cumprod()
    max_drawdown = (wealth / wealth.cummax() - 1).min()
    return pd.Series({
        "Ann. return": ann_return,
        "Ann. vol": ann_vol,
        "Sharpe": sharpe,
        "Max drawdown": max_drawdown
    })

stats = pd.DataFrame({
    "Equal Weight": performance_statistics(equal_returns),
    "ERC": performance_statistics(erc_returns)
}).T

stats


## 7. A simple DCC intuition

A full DCC model estimates univariate GARCH filters first, then a dynamic correlation equation. The following EWMA covariance filter is not a DCC estimator, but it gives the right intuition: correlations are state variables, not constants.


In [ ]:
def ewma_correlations(X, lam=0.97, burn_in=252):
    X = X.values
    S = np.cov(X[:burn_in].T)
    output = []
    dates = []
    for t in range(burn_in, len(X)):
        x = X[t - 1:t].T
        S = lam * S + (1 - lam) * (x @ x.T)
        d = np.sqrt(np.diag(S))
        C = S / np.outer(d, d)
        output.append(C.copy())
        dates.append(returns.index[t])
    return pd.DatetimeIndex(dates), output

selected = returns[["S&P500", "Eurostoxx 50", "Gold", "US IG Bonds"]]
dates, corr_matrices = ewma_correlations(selected, lam=0.97)

ewma_corr = pd.DataFrame({
    "S&P500 / Eurostoxx 50": [C[0, 1] for C in corr_matrices],
    "S&P500 / Gold": [C[0, 2] for C in corr_matrices],
    "S&P500 / US IG Bonds": [C[0, 3] for C in corr_matrices]
}, index=dates)

fig, ax = plt.subplots(figsize=(12, 6))
ewma_corr.plot(ax=ax)
ax.set_title("EWMA dynamic correlations: a simple DCC intuition")
ax.set_ylabel("Correlation")
ax.axhline(0, linewidth=1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. DCC-implied empirical betas

Engle's DCC framework is useful because it produces a conditional covariance matrix $H_t$. From this object, empirical betas are obtained directly:

$$
eta_{i|m,t} = rac{h_{im,t}}{h_{mm,t}} = ho_{im,t}rac{\sigma_{i,t}}{\sigma_{m,t}}.
$$

This is not a rolling OLS regression. It is a beta derived from the DCC covariance matrix.


In [ ]:
def dcc_filter(ret, lam_vol=0.94, alpha=0.02, beta=0.97, burn_in=252):
    """Transparent DCC-style filter.

    The univariate volatilities are EWMA filters and the correlation matrix follows the
    Engle DCC recursion. The parameters are fixed for teaching purposes.
    """
    X = ret.values.astype(float)
    T, n = X.shape
    mu = X[:burn_in].mean(axis=0)
    eps = X - mu

    h = np.full((T, n), np.nan)
    h[burn_in - 1] = eps[:burn_in].var(axis=0, ddof=1)
    for t in range(burn_in, T):
        h[t] = lam_vol * h[t - 1] + (1 - lam_vol) * (eps[t - 1] ** 2)

    z = eps / np.sqrt(h)
    z[:burn_in] = np.nan
    Qbar = np.cov(z[burn_in:].T)
    Q = Qbar.copy()

    H_list = []
    R_list = []
    dates = []
    for t in range(burn_in + 1, T):
        u = z[t - 1][:, None]
        Q = (1 - alpha - beta) * Qbar + alpha * (u @ u.T) + beta * Q
        dQ = np.sqrt(np.diag(Q))
        R = Q / np.outer(dQ, dQ)
        R = np.clip(R, -0.999, 0.999)
        np.fill_diagonal(R, 1.0)

        D = np.diag(np.sqrt(h[t]))
        H = D @ R @ D * ANNUALIZATION
        H_list.append(H)
        R_list.append(R.copy())
        dates.append(ret.index[t])

    return pd.DatetimeIndex(dates), np.array(H_list), np.array(R_list), pd.DataFrame(h, index=ret.index, columns=ret.columns)

beta_dates, Hs, Rs, h = dcc_filter(returns, lam_vol=0.94, alpha=0.02, beta=0.97, burn_in=252)

market = "S&P500"
m = assets.index(market)
betas_to_sp500 = pd.DataFrame({assets[i]: Hs[:, i, m] / Hs[:, m, m] for i in range(len(assets))}, index=beta_dates)
betas_to_sp500.tail()


In [ ]:
selected_betas = ["Eurostoxx 50", "Hang Seng", "Gold", "US IG Bonds", "US HY Bonds"]

fig, ax = plt.subplots(figsize=(12, 6))
betas_to_sp500[selected_betas].rolling(63).mean().plot(ax=ax)
ax.axhline(1, linestyle="--", linewidth=1)
ax.axhline(0, linestyle=":", linewidth=1)
ax.set_title("DCC-implied empirical betas to S&P 500")
ax.set_ylabel("Conditional beta")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


The decomposition below shows that a beta changes for two reasons: the conditional correlation can change and the relative volatility can change.


In [ ]:
j = assets.index("Eurostoxx 50")
vol_ratio = np.sqrt(Hs[:, j, j]) / np.sqrt(Hs[:, m, m])
rho = Rs[:, j, m]
beta_estoxx = rho * vol_ratio

beta_decomp = pd.DataFrame({
    "Beta": beta_estoxx,
    "DCC correlation": rho,
    "Volatility ratio": vol_ratio
}, index=beta_dates)

fig, ax = plt.subplots(figsize=(12, 6))
beta_decomp.rolling(63).mean().plot(ax=ax)
ax.axhline(0, linestyle=":", linewidth=1)
ax.set_title("Decomposing the Eurostoxx 50 beta to S&P 500")
ax.set_ylabel("Level")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", ncol=3, fontsize=8)
plt.tight_layout()
plt.show()


## 9. ERC portfolio betas to each asset

For a portfolio with weights $w_t$, the conditional beta of the portfolio to asset $j$ is

$$
eta_{p|j,t}=rac{\mathrm{Cov}_t(r_{p,t}, r_{j,t})}{\mathrm{Var}_t(r_{j,t})}
=rac{w_t'H_t e_j}{h_{jj,t}}.
$$

This is a useful complement to risk contributions: risk contributions tell us where portfolio volatility comes from; empirical betas tell us how the portfolio co-moves with each asset under the DCC covariance matrix.


In [ ]:
daily_weights = weights.reindex(returns.index, method="ffill")
daily_weights = daily_weights.loc[beta_dates].ffill().dropna()

mask = np.isin(beta_dates, daily_weights.index)
H_aligned = Hs[mask]
dates_aligned = beta_dates[mask]
W = daily_weights.loc[dates_aligned].values

erc_asset_betas = []
for k, H in enumerate(H_aligned):
    w = W[k]
    cov_p_assets = w @ H
    beta_p = cov_p_assets / np.diag(H)
    erc_asset_betas.append(beta_p)

erc_asset_betas = pd.DataFrame(erc_asset_betas, index=dates_aligned, columns=assets)

fig, ax = plt.subplots(figsize=(12, 6))
erc_asset_betas.rolling(63).mean().plot(ax=ax)
ax.axhline(0, linestyle=":", linewidth=1)
ax.set_title("DCC-implied beta of the ERC portfolio to each asset")
ax.set_ylabel(r"$eta_{p|j,t}=Cov_t(r_p,r_j)/Var_t(r_j)$")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
beta_summary = pd.DataFrame({
    "Average beta": erc_asset_betas.mean(),
    "Latest beta": erc_asset_betas.iloc[-1],
    "Min beta": erc_asset_betas.min(),
    "Max beta": erc_asset_betas.max()
})

beta_summary.round(2)


## 10. Exercises

1. Change the rolling covariance window from 2 years to 1 year and 3 years. How stable are the ERC weights?
2. Replace the rolling covariance matrix with the DCC covariance matrix. What changes?
3. Compare ERC to minimum variance and maximum diversification portfolios.
4. Redo the analysis after excluding commodities. What happens to drawdowns and risk contributions?
5. Use the dynamic correlation series to discuss when diversification is most fragile.
6. For the DCC beta application, change the benchmark from S&P 500 to Eurostoxx 50. Which assets have the most unstable beta?
7. Compare the ERC portfolio beta to US IG bonds and US HY bonds. How does this help interpret the backtest?
